In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import optuna
from pathlib import Path

from src.dominick import DominickDataLoader
from src.dominick.multiproduct_builder import MultiProductBuilder
from src.nn.data import ColumnEncoder, DataLoaderFactory, SplineBuilder
from src.nn.spline import MultiCubicSplineBasis
from src.nn.models import IntegrableDemandHead, ICDN
from src.nn.loss import ElasticityLoss
from src.multiproduct import MultiProductDataset, MultiProductContextEmbeddings

/home/thebigmonster/Github/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [4]:
# ── Datos ──────────────────────────────────────────────────────────
N_UPCS        = 5
TRAIN_FRAC    = 0.8
SMOOTH_WINDOW = 8

BETA_EDA = -1.74

# ── Entrenamiento (reducido para que el search sea rápido) ─────────
N_EPOCHS_P0 = 350
N_EPOCHS_P1 = 350
N_EPOCHS_P2 = 400
PATIENCE    = 30
ES_PATIENCE = 30

# ── Checkpoints ────────────────────────────────────────────────────
CKPT_DIR = Path("../results/checkpoints/hparam")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
loader = DominickDataLoader()
df     = loader.load("elasticity_dataset.csv")
print(f"Dataset shape: {df.shape}")

encoder = ColumnEncoder()
_, store_cats = encoder.factorize(df, "store_code", sort=True)
_, week_cats  = encoder.factorize(df, "week_id",    sort=True)
n_stores = len(store_cats)
n_weeks  = len(week_cats)
print(f"Tiendas: {n_stores}  |  Semanas: {n_weeks}")

sorted_weeks   = sorted(df["week_id"].unique())
week_threshold = sorted_weeks[int(len(sorted_weeks) * TRAIN_FRAC)]
train_df_raw   = df[df["week_id"] < week_threshold].copy()

mp_builder = MultiProductBuilder()
mp_builder.fit(train_df_raw, n=N_UPCS)
full_wide  = mp_builder.transform(df)
train_wide = full_wide[full_wide["week_id"] < week_threshold].copy()
val_wide   = full_wide[full_wide["week_id"] >= week_threshold].copy()
n_upcs     = mp_builder.n

store_map = {v: i for i, v in enumerate(store_cats)}
week_map  = {v: i for i, v in enumerate(week_cats)}
for w in [train_wide, val_wide]:
    w["store_code"] = w["store_code"].map(store_map)
    w["week_id"]    = w["week_id"].map(week_map)

print(f"Train: {len(train_wide):,}  |  Val: {len(val_wide):,}")
print(f"UPCs seleccionados: {n_upcs}")

Dataset shape: (1966147, 10)
Tiendas: 89  |  Semanas: 302
Train: 1,205  |  Val: 305
UPCs seleccionados: 5


In [6]:
train_wide_s = train_wide.sort_values(["store_code", "week_id"]).copy()
val_wide_s   = val_wide.sort_values(["store_code", "week_id"]).copy()

for i in range(n_upcs):
    col = f"log_liters_{i}"
    for df_w in [train_wide_s, val_wide_s]:
        df_w[col] = (
            df_w.groupby("store_code")[col]
            .transform(lambda s: s.rolling(window=SMOOTH_WINDOW, min_periods=1).mean())
        )

loader_factory = DataLoaderFactory(num_workers=0, pin_memory=True)

train_ds_p0     = MultiProductDataset(train_wide_s, n=n_upcs)
val_ds_p0       = MultiProductDataset(val_wide_s,   n=n_upcs)
train_loader_p0 = loader_factory.create_train_loader(train_ds_p0, shuffle=True,  drop_last=True)
val_loader_p0   = loader_factory.create_eval_loader(val_ds_p0,   shuffle=False)

train_ds     = MultiProductDataset(train_wide, n=n_upcs)
val_ds       = MultiProductDataset(val_wide,   n=n_upcs)
train_loader = loader_factory.create_train_loader(train_ds, shuffle=True,  drop_last=True)
val_loader   = loader_factory.create_eval_loader(val_ds,   shuffle=False)

# Límites de semana (fijos, independientes de N_KNOTS)
week_min = float(min(train_wide["week_id"].min(), val_wide["week_id"].min()))
week_max = float(max(train_wide["week_id"].max(), val_wide["week_id"].max()))

print("DataLoaders y límites de semana listos")

DataLoaders y límites de semana listos


In [7]:
def run_training(model, train_loader, val_loader, loss_fn,
                 optimizer, scheduler, n_epochs, es_patience,
                 ckpt_path, device, phase_name="", verbose=False):

    best_val_loss = float("inf")
    no_improve    = 0
    scaler        = torch.amp.GradScaler("cuda") if device == "cuda" else None

    for epoch in range(n_epochs):
        # ── Train ──────────────────────────────────────────────────
        model.train()
        total_loss, total_denom = 0.0, 0.0

        for batch in train_loader:
            batch    = {k: v.to(device) for k, v in batch.items()}
            y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
            obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)

            optimizer.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    y_hat, eps_hat, aux = model(batch, return_parts=True)
                    loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                        aux["w"], aux["ddBx"], aux["u"],
                                        aux["Bx"], aux["IBx"],
                                        model.head.param_head._pairs)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                y_hat, eps_hat, aux = model(batch, return_parts=True)
                loss, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                    aux["w"], aux["ddBx"], aux["u"],
                                    aux["Bx"], aux["IBx"],
                                    model.head.param_head._pairs)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            denom        = obs_mask.sum().item()
            total_loss  += logs["loss"].item() * denom
            total_denom += denom

        # ── Val ────────────────────────────────────────────────────
        model.eval()
        val_loss_sum, val_denom = 0.0, 0.0

        with torch.no_grad():
            for batch in val_loader:
                batch    = {k: v.to(device) for k, v in batch.items()}
                y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
                obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)

                y_hat, eps_hat, aux = model(batch, return_parts=True)
                _, logs = loss_fn(y_hat, y_true, eps_hat, obs_mask,
                                  aux["w"], aux["ddBx"], aux["u"],
                                  aux["Bx"], aux["IBx"],
                                  model.head.param_head._pairs)
                denom        = obs_mask.sum().item()
                val_loss_sum += logs["loss"].item() * denom
                val_denom    += denom

        val_loss = val_loss_sum / max(val_denom, 1.0)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        if verbose and ((epoch + 1) % 50 == 0 or no_improve == 0):
            print(f"  [{phase_name}] Epoch {epoch+1}  val={val_loss:.4f}")

        if no_improve >= es_patience:
            if verbose:
                print(f"  [{phase_name}] Early stopping en época {epoch+1}")
            break

    return best_val_loss

print("run_training definida")

run_training definida


In [8]:
HIDDEN_OPTIONS = {
    "64_32":      (64, 32),
    "128_64_32":  (128, 64, 32),
    "64_32_16":   (64, 32, 16),
    "128_64":     (128, 64),
}

def compute_global_r2(model, val_loader, device):
    """R² global en log-liters sobre todas las observaciones de validación."""
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch    = {k: v.to(device) for k, v in batch.items()}
            y_true   = torch.stack([batch[f"log_liters_{i}"] for i in range(model.n)], dim=1)
            obs_mask = torch.stack([batch[f"obs_mask_{i}"]   for i in range(model.n)], dim=1)
            y_hat, _, _ = model(batch, return_parts=True)

            mask = obs_mask.bool()
            all_true.append(y_true[mask].cpu())
            all_pred.append(y_hat[mask].cpu())

    y_true_all = torch.cat(all_true)
    y_pred_all = torch.cat(all_pred)

    ss_res = ((y_true_all - y_pred_all) ** 2).sum()
    ss_tot = ((y_true_all - y_true_all.mean()) ** 2).sum()
    return float(1.0 - ss_res / ss_tot)


def compute_elasticity_score(model, val_loader, device,
                              elast_min=-5.0, elast_max=-0.1):
    """
    Score de coherencia de elasticidades propias (eps_hat) en validación.
    Combina:
      - fracción de elasticidades dentro del rango económico [elast_min, elast_max]
      - penalización si la mediana se aleja mucho del prior BETA_EDA
    Devuelve un valor entre 0 y 1 (mayor = más coherente).
    """
    model.eval()
    all_elast = []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            _, eps_hat, _ = model(batch, return_parts=True)
            # eps_hat: (B, n_upcs) — elasticidades propias de cada producto
            all_elast.append(eps_hat.cpu())

    elast = torch.cat(all_elast, dim=0).flatten().numpy()

    # Fracción en rango económico razonable
    in_range = float(((elast >= elast_min) & (elast <= elast_max)).mean())

    # Penalización por desviación de la mediana respecto al prior EDA
    median_e = float(np.median(elast))
    deviation = max(0.0, abs(median_e - BETA_EDA) - 0.3)   # tolerancia ±0.3
    prior_penalty = min(deviation / abs(BETA_EDA), 1.0)     # normalizado a [0,1]

    score = in_range * (1.0 - prior_penalty)
    return float(score), float(median_e), float(in_range)


def build_and_train(params, trial_id=0):
    """
    Entrena las 3 fases con los hiperparámetros dados.
    Devuelve (r2_global, elast_score) para optimización multi-objetivo.
    """
    fourier_harm     = params["FOURIER_HARM"]
    n_knots          = params["N_KNOTS"]
    hidden           = HIDDEN_OPTIONS[params["HIDDEN_KEY"]] 
    dropout          = params["DROPOUT"]
    d_store          = params.get("D_STORE", 16)
    act              = params.get("ACT", "gelu")
    lr_p0            = params["LR_P0"]
    lr_p1            = params["LR_P1"]
    lr_p2            = params["LR_P2"]
    lambda_smooth_p2 = params["LAMBDA_SMOOTH_P2"]
    lambda_pos_p2    = params["LAMBDA_POS_P2"]

    ckpt_p0 = CKPT_DIR / f"trial{trial_id}_phase0.pt"
    ckpt_p1 = CKPT_DIR / f"trial{trial_id}_phase1.pt"
    ckpt_p2 = CKPT_DIR / f"trial{trial_id}_phase2.pt"

    # ── Splines ────────────────────────────────────────────────────
    builder = SplineBuilder()
    spline_configs = []
    for i in range(n_upcs):
        x_i    = train_wide[f"log_price_{i}"].values
        config = builder.build_from_data(x_i, n_knots=n_knots, q_min=0.05, q_max=0.95)
        spline_configs.append(config)

    knots = torch.stack([cfg["knots"] for cfg in spline_configs], dim=0)
    shift = torch.tensor([cfg["mean"]  for cfg in spline_configs])
    scale = torch.tensor([cfg["std"]   for cfg in spline_configs])
    price_splines = MultiCubicSplineBasis(knots=knots, shift=shift, scale=scale)

    # ── Context builder ─────────────────────────────────────────────
    cb = MultiProductContextEmbeddings(
        n=n_upcs,
        n_stores=n_stores,
        d_store=d_store,
        fourier_period=52.0,
        fourier_harmonics=fourier_harm,
        include_trend=True,
        week_min=week_min,
        week_max=week_max,
        regressors={"lag_y", "lag_y_52", "rolling_mean_y"},
    )

    def make_model(enforce_negative_beta, use_cross):
        head = IntegrableDemandHead(
            context_dim=cb.out_dim,
            K_splines=n_knots,
            n=n_upcs,
            hidden=hidden,
            act=act,
            dropout=dropout,
            use_cross=use_cross,
            enforce_negative_beta=enforce_negative_beta,
        )
        return ICDN(
            context_builder=cb,
            price_splines=price_splines,
            head=head,
            n=n_upcs,
        ).to(device)

    # ── FASE 0 ─────────────────────────────────────────────────────
    m0 = make_model(enforce_negative_beta=True, use_cross=False)
    with torch.no_grad():
        m0.head.param_head.head_w.weight.zero_()
        m0.head.param_head.head_w.bias.zero_()
    m0.head.param_head.head_w.weight.requires_grad_(False)
    m0.head.param_head.head_w.bias.requires_grad_(False)

    beta_raw_init = torch.log(
        torch.exp(torch.tensor(-BETA_EDA, dtype=torch.float32)) - 1.0
    )
    with torch.no_grad():
        m0.head.param_head.head_beta.weight.zero_()
        m0.head.param_head.head_beta.bias.fill_(beta_raw_init)

    loss_p0 = ElasticityLoss(huber_delta=1.0, lambda_smooth=0.0, lambda_pos=0.0, reduction="none")
    opt_p0  = torch.optim.AdamW(m0.parameters(), lr=lr_p0, weight_decay=1e-5)
    sch_p0  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p0, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5)
    run_training(m0, train_loader_p0, val_loader_p0, loss_p0,
                 opt_p0, sch_p0, N_EPOCHS_P0, ES_PATIENCE, ckpt_p0, device, "P0")

    # ── FASE 1 ─────────────────────────────────────────────────────
    m1 = make_model(enforce_negative_beta=True, use_cross=False)
    m1.load_state_dict(torch.load(ckpt_p0, map_location=device))
    m1.head.param_head.head_w.weight.requires_grad_(False)
    m1.head.param_head.head_w.bias.requires_grad_(False)

    loss_p1 = ElasticityLoss(huber_delta=1.0, lambda_smooth=0.0, lambda_pos=0.0, reduction="none")
    opt_p1  = torch.optim.AdamW(m1.parameters(), lr=lr_p1, weight_decay=1e-5)
    sch_p1  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p1, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5)
    run_training(m1, train_loader, val_loader, loss_p1,
                 opt_p1, sch_p1, N_EPOCHS_P1, ES_PATIENCE, ckpt_p1, device, "P1")

    # ── FASE 2 ─────────────────────────────────────────────────────
    m2 = make_model(enforce_negative_beta=True, use_cross=True)
    state = torch.load(ckpt_p1, map_location=device)
    state.pop("head.param_head._pairs", None)
    m2.load_state_dict(state, strict=False)

    m2.head.param_head.head_w.weight.requires_grad_(True)
    m2.head.param_head.head_w.bias.requires_grad_(True)
    with torch.no_grad():
        m2.head.param_head.head_cross.weight.zero_()
        m2.head.param_head.head_cross.bias.zero_()

    loss_p2 = ElasticityLoss(
        huber_delta=1.0,
        lambda_smooth=lambda_smooth_p2,
        lambda_pos=lambda_pos_p2,
        reduction="none",
    )
    opt_p2 = torch.optim.AdamW(m2.parameters(), lr=lr_p2, weight_decay=1e-5)
    sch_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt_p2, mode="min", factor=0.5, patience=PATIENCE, min_lr=1e-5)
    run_training(m2, train_loader, val_loader, loss_p2,
                 opt_p2, sch_p2, N_EPOCHS_P2, ES_PATIENCE, ckpt_p2, device, "P2")

    # ── Métricas finales sobre el mejor checkpoint de Fase 2 ───────
    m2.load_state_dict(torch.load(ckpt_p2, map_location=device))

    r2                        = compute_global_r2(m2, val_loader, device)
    elast_score, median_e, in_range = compute_elasticity_score(m2, val_loader, device)

    print(f"  R²={r2:.4f}  |  elast_score={elast_score:.4f}"
          f"  (mediana={median_e:.3f}, en_rango={in_range:.2%})")

    # Limpiar checkpoints intermedios
    ckpt_p0.unlink(missing_ok=True)
    ckpt_p1.unlink(missing_ok=True)

    return r2, elast_score

print("Métricas y build_and_train definidas")

Métricas y build_and_train definidas


In [9]:
def objective(trial):
    params = {
        "FOURIER_HARM":     trial.suggest_int("FOURIER_HARM", 3, 12),
        "N_KNOTS":          trial.suggest_int("N_KNOTS", 2, 6),
        "HIDDEN_KEY":       trial.suggest_categorical("HIDDEN_KEY", list(HIDDEN_OPTIONS.keys())), 
        "DROPOUT":          trial.suggest_float("DROPOUT", 0.0, 0.3),
        "LR_P0":            trial.suggest_float("LR_P0",  1e-4, 1e-2, log=True),
        "LR_P1":            trial.suggest_float("LR_P1",  1e-5, 5e-3, log=True),
        "LR_P2":            trial.suggest_float("LR_P2",  1e-5, 1e-3, log=True),
        "LAMBDA_SMOOTH_P2": trial.suggest_float("LAMBDA_SMOOTH_P2", 1e-5, 1e-2, log=True),
        "LAMBDA_POS_P2":    trial.suggest_float("LAMBDA_POS_P2",    0.05, 0.5),
    }

    print(f"\n{'='*60}")
    print(f"Trial {trial.number}")
    for k, v in params.items():
        print(f"  {k}: {v}")
    print(f"{'='*60}")

    r2, elast_score = build_and_train(params, trial_id=trial.number)
    return r2, elast_score   # Optuna recibe los DOS valores

In [10]:
study = optuna.create_study(
    directions=["maximize", "maximize"],   # maximizar R² Y elast_score
    study_name="hparam_pareto",
    storage="sqlite:///../results/hparam_pareto.db",
    load_if_exists=True,
)

study.optimize(objective, n_trials=25)

print(f"\nTrials completados: {len(study.trials)}")
print(f"Trials Pareto-óptimos: {len(study.best_trials)}")

[I 2026-03-03 12:57:56,524] A new study created in RDB with name: hparam_pareto



Trial 0
  FOURIER_HARM: 3
  N_KNOTS: 3
  HIDDEN_KEY: 128_64_32
  DROPOUT: 0.11324229176062166
  LR_P0: 0.00013635620594767135
  LR_P1: 0.0001722183201879389
  LR_P2: 1.1662505910801746e-05
  LAMBDA_SMOOTH_P2: 0.009197369020945981
  LAMBDA_POS_P2: 0.35865787374766533


[I 2026-03-03 12:59:00,401] Trial 0 finished with values: [0.4028201699256897, 0.26051862982452895] and parameters: {'FOURIER_HARM': 3, 'N_KNOTS': 3, 'HIDDEN_KEY': '128_64_32', 'DROPOUT': 0.11324229176062166, 'LR_P0': 0.00013635620594767135, 'LR_P1': 0.0001722183201879389, 'LR_P2': 1.1662505910801746e-05, 'LAMBDA_SMOOTH_P2': 0.009197369020945981, 'LAMBDA_POS_P2': 0.35865787374766533}.


  R²=0.4028  |  elast_score=0.2605  (mediana=-0.352, en_rango=69.57%)

Trial 1
  FOURIER_HARM: 7
  N_KNOTS: 3
  HIDDEN_KEY: 64_32_16
  DROPOUT: 0.27131277604704096
  LR_P0: 0.0004883699463944475
  LR_P1: 3.7543518219220075e-05
  LR_P2: 0.0002228186486621899
  LAMBDA_SMOOTH_P2: 0.0026739478631022393
  LAMBDA_POS_P2: 0.2839153774045417


[I 2026-03-03 12:59:19,674] Trial 1 finished with values: [-0.9721840620040894, 0.23150292538149964] and parameters: {'FOURIER_HARM': 7, 'N_KNOTS': 3, 'HIDDEN_KEY': '64_32_16', 'DROPOUT': 0.27131277604704096, 'LR_P0': 0.0004883699463944475, 'LR_P1': 3.7543518219220075e-05, 'LR_P2': 0.0002228186486621899, 'LAMBDA_SMOOTH_P2': 0.0026739478631022393, 'LAMBDA_POS_P2': 0.2839153774045417}.


...

  R²=-1.9396  |  elast_score=0.0000  (mediana=-7.508, en_rango=36.52%)

Trial 23
  FOURIER_HARM: 8
  N_KNOTS: 5
  HIDDEN_KEY: 128_64_32
  DROPOUT: 0.012617338669275646
  LR_P0: 0.0023084634739164896
  LR_P1: 1.5195584843769103e-05
  LR_P2: 0.00013504270213129252
  LAMBDA_SMOOTH_P2: 1.9867429625683223e-05
  LAMBDA_POS_P2: 0.42620875223218835


[I 2026-03-03 13:09:52,471] Trial 23 finished with values: [-0.3995532989501953, 0.0] and parameters: {'FOURIER_HARM': 8, 'N_KNOTS': 5, 'HIDDEN_KEY': '128_64_32', 'DROPOUT': 0.012617338669275646, 'LR_P0': 0.0023084634739164896, 'LR_P1': 1.5195584843769103e-05, 'LR_P2': 0.00013504270213129252, 'LAMBDA_SMOOTH_P2': 1.9867429625683223e-05, 'LAMBDA_POS_P2': 0.42620875223218835}.


  R²=-0.3996  |  elast_score=0.0000  (mediana=-5.738, en_rango=24.33%)

Trial 24
  FOURIER_HARM: 10
  N_KNOTS: 4
  HIDDEN_KEY: 128_64_32
  DROPOUT: 0.10147656334527651
  LR_P0: 0.00031998652602546453
  LR_P1: 0.00022247610724963337
  LR_P2: 0.00016028598344201608
  LAMBDA_SMOOTH_P2: 0.001412800399699337
  LAMBDA_POS_P2: 0.30602445645764925


[I 2026-03-03 13:10:23,856] Trial 24 finished with values: [0.015348494052886963, 0.03184200319076677] and parameters: {'FOURIER_HARM': 10, 'N_KNOTS': 4, 'HIDDEN_KEY': '128_64_32', 'DROPOUT': 0.10147656334527651, 'LR_P0': 0.00031998652602546453, 'LR_P1': 0.00022247610724963337, 'LR_P2': 0.00016028598344201608, 'LAMBDA_SMOOTH_P2': 0.001412800399699337, 'LAMBDA_POS_P2': 0.30602445645764925}.


  R²=0.0153  |  elast_score=0.0318  (mediana=-0.048, en_rango=15.93%)

Trials completados: 25
Trials Pareto-óptimos: 2


In [11]:
print(f"\n{'='*70}")
print(f"{'Trial':>6}  {'R²':>8}  {'ElastScore':>11}  Parámetros")
print(f"{'='*70}")

pareto = sorted(study.best_trials, key=lambda t: t.values[0], reverse=True)
for t in pareto:
    r2_val = t.values[0]
    es_val = t.values[1]
    params_str = ", ".join(f"{k.replace('params_','')}={v}"
                           for k, v in t.params.items())
    print(f"  {t.number:4d}  {r2_val:8.4f}  {es_val:11.4f}  {params_str}")


 Trial        R²   ElastScore  Parámetros
     4    0.4102       0.5254  FOURIER_HARM=3, N_KNOTS=3, HIDDEN_KEY=128_64, DROPOUT=0.17788977944370002, LR_P0=0.0014839072616042034, LR_P1=0.003820907469730344, LR_P2=3.427869269296773e-05, LAMBDA_SMOOTH_P2=0.0006091978303293139, LAMBDA_POS_P2=0.1500621889886809
    21    0.2920       0.5889  FOURIER_HARM=9, N_KNOTS=3, HIDDEN_KEY=128_64, DROPOUT=0.2524714773097736, LR_P0=0.007276683262369221, LR_P1=3.2774172875982194e-05, LR_P2=1.6741900874229782e-05, LAMBDA_SMOOTH_P2=0.00017673008603011443, LAMBDA_POS_P2=0.3940381513004254


In [12]:
df_trials = study.trials_dataframe()
param_cols = [c for c in df_trials.columns if c.startswith("params_")]
val_cols   = [c for c in df_trials.columns if c.startswith("values_")]

df_show = (df_trials[["number"] + val_cols + param_cols]
           .rename(columns={"values_0": "R2", "values_1": "elast_score"})
           .dropna(subset=["R2"])
           .sort_values("R2", ascending=False))

print(df_show.head(15).to_string(index=False))

 number        R2  elast_score  params_DROPOUT  params_FOURIER_HARM params_HIDDEN_KEY  params_LAMBDA_POS_P2  params_LAMBDA_SMOOTH_P2  params_LR_P0  params_LR_P1  params_LR_P2  params_N_KNOTS
      4  0.410214     0.525406        0.177890                    3            128_64              0.150062                 0.000609      0.001484      0.003821      0.000034               3
      3  0.409214     0.227755        0.283048                    3            128_64              0.367853                 0.007185      0.000935      0.000068      0.000012               4
      0  0.402820     0.260519        0.113242                    3         128_64_32              0.358658                 0.009197      0.000136      0.000172      0.000012               3
     18  0.396509     0.387928        0.180041                    6             64_32              0.255906                 0.000015      0.001239      0.001256      0.000060               2
     15  0.303862     0.461228        0.01505